# Laboratorio #9 — Embeddings de Palabras con Word2Vec, FastText, PPMI y SVD
Natural Language Processing, Universidad Francisco Marroquín
Autor: George Albadr
Nota: el PDF de instrucciones trae por error el título Laboratorio #8 Embedding; en la secuencia del curso es el Laboratorio #9.

---
### Objetivo
Construir y comparar representaciones vectoriales de palabras en español, en siete partes:
1. Preparación de los corpus de noticias y del Quijote, con una normalización compatible con fastText SUC.
2. Co-ocurrencia, PPMI y SVD sobre el corpus de noticias.
3. Investigación teórica sobre CBOW, skip-gram, negative sampling, GloVe, fastText y nn.Embedding.
4. Entrenamiento de Word2Vec y FastText con variantes, y comparación entre noticias y Quijote.
5. Similitud, vecinos y analogías en cuatro modelos.
6. Visualización con PCA y t-SNE.
7. Análisis y respuestas finales.

## 1. Preparación de los corpus

### Normalización del texto
La hipótesis distribucional dice que una palabra se conoce por sus vecinas, así que cada decisión de limpieza cambia el espacio que se aprende.
1. Minúsculas: fastText SUC está en minúsculas, y así El y el no se separan en dos vectores.
2. Tildes y eñes se conservan: año no es ano, y término no es terminó. SUC conserva los acentos, y quitarlos rompería la comparación de la sección 5.
3. Puntuación: primero se separa en oraciones con sent_tokenize y después se extraen solo tokens alfabéticos, para que comas y puntos no actúen como vecinos falsos.
4. Stopwords: no se quitan. Son el andamio de la ventana; sin ellas, palabras separadas por cinco o seis términos quedan pegadas y el contexto se deforma.
5. Lematización: no se aplica. Juntar gobiernos con gobernó borra el número y el tiempo verbal, justo lo que permite aprender plurales y conjugaciones.

In [1]:
import os
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import sent_tokenize
from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE

from gensim.models import Word2Vec, FastText
from gensim.models.keyedvectors import KeyedVectors

# Helper para resolver rutas relativas robustamente (tanto desde raíz como dentro de Lab#9)
def resolver_ruta(nombre, alternativas=None):
    candidatos = [nombre]
    if alternativas:
        candidatos.extend(alternativas)
    candidatos.extend([
        os.path.join(".", nombre),
        os.path.join("..", nombre),
        os.path.join("Lab#9", nombre),
        os.path.join("..", "Lab#1", nombre),
        os.path.join("Lab#1", nombre),
    ])
    for c in candidatos:
        if os.path.exists(c):
            return c
    return nombre

def ruta_guardado(nombre):
    if os.path.exists("Lab#9") and os.path.isdir("Lab#9"):
        return os.path.join("Lab#9", nombre)
    return nombre

# Descargas de soporte nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 1. Cargar Spanish News Classification
ruta_news = resolver_ruta("df_total.csv", ["Lab#1/df_total.csv", "../Lab#1/df_total.csv", "Lab#9/df_total.csv"])
print(f"Cargando noticias desde: {ruta_news}")
df_news = pd.read_csv(ruta_news)
news_texts = df_news['news'].dropna().tolist()

# 2. Cargar Don Quijote
quijote_path = os.path.expanduser("~/.cache/kagglehub/datasets/manuelmaaf97/quijote/versions/1/don-quijote.txt")
if not os.path.exists(quijote_path):
    import kagglehub
    quijote_dir = kagglehub.dataset_download('manuelmaaf97/quijote')
    quijote_path = os.path.join(quijote_dir, "don-quijote.txt")

with open(quijote_path, encoding='utf-8') as f:
    quijote_raw = f.read()

# Recortar licencias y prefacios legales
ini = quijote_raw.find("El ingenioso hidalgo don Quijote de la Mancha")
fin = quijote_raw.rfind("Fin") + len("Fin")
quijote_text = quijote_raw[ini:fin]

def normalizar_oracion(texto_oracion):
    # Palabras alfabéticas en minúsculas conservando tildes y eñes
    return re.findall(r'[a-záéíóúüñ]+', texto_oracion.lower())

# Procesar Noticias en lista de listas de tokens
oraciones_noticias = []
for doc in news_texts:
    doc_proc = re.sub(r'([.?!])([A-ZÁÉÍÓÚÑ])', r'\1 \2', doc)
    for s in sent_tokenize(doc_proc, language='spanish'):
        toks = normalizar_oracion(s)
        if len(toks) > 1:
            oraciones_noticias.append(toks)

# Procesar Quijote en lista de listas de tokens
oraciones_quijote = []
quijote_proc = re.sub(r'([.?!])([A-ZÁÉÍÓÚÑ])', r'\1 \2', quijote_text)
for s in sent_tokenize(quijote_proc, language='spanish'):
    toks = normalizar_oracion(s)
    if len(toks) > 1:
        oraciones_quijote.append(toks)

tokens_noticias_total = sum(len(o) for o in oraciones_noticias)
vocab_noticias = set(w for o in oraciones_noticias for w in o)

tokens_quijote_total = sum(len(o) for o in oraciones_quijote)
vocab_quijote = set(w for o in oraciones_quijote for w in o)

tabla_corpus = pd.DataFrame([
    {
        "Corpus": "Spanish News Classification",
        "Documentos": len(news_texts),
        "Oraciones": len(oraciones_noticias),
        "Tokens": tokens_noticias_total,
        "Tipos (Vocabulario)": len(vocab_noticias)
    },
    {
        "Corpus": "Don Quijote de la Mancha",
        "Documentos": 1,
        "Oraciones": len(oraciones_quijote),
        "Tokens": tokens_quijote_total,
        "Tipos (Vocabulario)": len(vocab_quijote)
    }
])
print("=== Estadísticas Generales de los Corpus ===")
display(tabla_corpus)


/Users/georgealbadr/Github/Natural_Language_Processing/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Cargando noticias desde: ../Lab#1/df_total.csv
=== Estadísticas Generales de los Corpus ===


,Corpus,Documentos,Oraciones,Tokens,Tipos (Vocabulario)
0,Spanish News Classification,1217,21769,622871,27759
1,Don Quijote de la Mancha,1,9471,381181,22933


### Frecuencia mínima en noticias, con min_count 1, 2 y 5
Filtrar palabras raras reduce mucho el vocabulario y descarta muy pocos tokens.

In [2]:
conteo_noticias = Counter(w for o in oraciones_noticias for w in o)
tabla_min_count = []
for mc in [1, 2, 5]:
    vocab_mc = {w for w, c in conteo_noticias.items() if c >= mc}
    tokens_dentro = sum(c for w, c in conteo_noticias.items() if c >= mc)
    tokens_fuera = tokens_noticias_total - tokens_dentro
    pct_fuera = (tokens_fuera / tokens_noticias_total) * 100.0
    tabla_min_count.append({
        "min_count": mc,
        "Tamaño Vocabulario (|V|)": len(vocab_mc),
        "Tokens dentro": tokens_dentro,
        "Tokens fuera (OOV)": tokens_fuera,
        "% Tokens fuera": round(pct_fuera, 2)
    })

df_min_count = pd.DataFrame(tabla_min_count)
print("=== Impacto del parámetro min_count en Noticias ===")
display(df_min_count)


=== Impacto del parámetro min_count en Noticias ===


,min_count,Tamaño Vocabulario (|V|),Tokens dentro,Tokens fuera (OOV),% Tokens fuera
0,1,27759,622871,0,0.00
1,2,16190,611302,11569,1.86
2,5,8543,590919,31952,5.13


### Pares skip-gram a mano con ventana 2
Se extraen con código propio los pares de palabra central y contexto de una oración de las noticias.

In [3]:
oracion_ejemplo = oraciones_noticias[0][:10]  # Primeras 10 palabras de la primera noticia

pares_mano = []
for i, w_central in enumerate(oracion_ejemplo):
    start = max(0, i - 2)
    end = min(len(oracion_ejemplo), i + 3)
    for j in range(start, end):
        if i != j:
            pares_mano.append((w_central, oracion_ejemplo[j]))

print(f"Oración seleccionada: {' '.join(oracion_ejemplo)}")
print(f"Total de pares generados (ventana ±2): {len(pares_mano)}\n")
print(f"{'Palabra Central (w)':22} -> {'Contexto (c)':20}")
print("-" * 45)
for p in pares_mano[:12]:
    print(f"{p[0]:22} -> {p[1]:20}")
print("... (mostrando primeros 12 de 34 pares)")


Oración seleccionada: durante el foro la banca articulador empresarial para el desarrollo
Total de pares generados (ventana ±2): 34

Palabra Central (w)    -> Contexto (c)        
---------------------------------------------
durante                -> el                  
durante                -> foro                
el                     -> durante             
el                     -> foro                
el                     -> la                  
foro                   -> durante             
foro                   -> el                  
foro                   -> la                  
foro                   -> banca               
la                     -> el                  
la                     -> foro                
la                     -> banca               
... (mostrando primeros 12 de 34 pares)


### Pares de entrenamiento y aprendizaje autosupervisado
Se cuentan los pares que produce todo el corpus de noticias con ventana 2 y con ventana 5, y se comparan con el número de tokens.

Word2Vec no necesita etiquetas porque el texto ya las trae: la palabra central es la entrada y sus vecinas son lo que debe predecir. Un corpus sin anotar de 622,871 tokens da más de 5.5 millones de ejemplos con ventana 5.

In [4]:
def contar_pares_skipgram(corpus_oraciones, ventana):
    total = 0
    for oracion in corpus_oraciones:
        L = len(oracion)
        if L < 2:
            continue
        for i in range(L):
            left = min(ventana, i)
            right = min(ventana, L - 1 - i)
            total += (left + right)
    return total

pares_v2 = contar_pares_skipgram(oraciones_noticias, 2)
pares_v5 = contar_pares_skipgram(oraciones_noticias, 5)

print(f"Total de tokens en noticias: {tokens_noticias_total:,}")
print(f"Pares skip-gram con ventana 2: {pares_v2:,} (ratio: {pares_v2/tokens_noticias_total:.2f} pares/token)")
print(f"Pares skip-gram con ventana 5: {pares_v5:,} (ratio: {pares_v5/tokens_noticias_total:.2f} pares/token)")


Total de tokens en noticias: 622,871
Pares skip-gram con ventana 2: 2,360,870 (ratio: 3.79 pares/token)
Pares skip-gram con ventana 5: 5,576,706 (ratio: 8.95 pares/token)


## 2. Co-ocurrencia, PPMI y SVD sobre noticias

### 2.1 Matriz de co-ocurrencia
Se construye la matriz palabra por palabra con ventana 2 para las 2,000 palabras más frecuentes.

In [5]:
top_2000_words = [w for w, c in conteo_noticias.most_common(2000)]
w2i = {w: i for i, w in enumerate(top_2000_words)}
i2w = {i: w for i, w in enumerate(top_2000_words)}
V_size = len(top_2000_words)

VENTANA_COOC = 2
cooc_matrix = np.zeros((V_size, V_size), dtype=np.float64)

for oracion in oraciones_noticias:
    indices = [w2i[w] for w in oracion if w in w2i]
    L = len(indices)
    for i in range(L):
        w_idx = indices[i]
        start = max(0, i - VENTANA_COOC)
        end = min(L, i + VENTANA_COOC + 1)
        for j in range(start, end):
            if i != j:
                cooc_matrix[w_idx, indices[j]] += 1.0

ceros = np.count_nonzero(cooc_matrix == 0)
total_celdas = V_size * V_size
pct_ceros = (ceros / total_celdas) * 100.0

print(f"Forma de la matriz de co-ocurrencia: {cooc_matrix.shape}")
print(f"Porcentaje de ceros (dispersión): {pct_ceros:.2f}% ({ceros:,} de {total_celdas:,} celdas son 0)")


Forma de la matriz de co-ocurrencia: (2000, 2000)
Porcentaje de ceros (dispersión): 92.88% (3,715,220 de 4,000,000 celdas son 0)


### Comparación con la matriz BoW del Laboratorio 2
La matriz de co-ocurrencia tiene 92.88% de ceros. La matriz BoW es documento por término, con textos cortos sobre un vocabulario enorme, y es más rala todavía. La de co-ocurrencia es más densa porque junta la evidencia de todos los documentos.

In [6]:
def sim_coseno_numpy(u, v):
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0.0 or norm_v == 0.0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

pares_test = [
    ("empresa", "mercado"),
    ("gobierno", "país"),
    ("banco", "financiero"),
    ("empresa", "caballo"),
    ("gobierno", "nube")
]

print("=== Similitud Coseno con Vectores Crudos de Co-ocurrencia ===")
for w1, w2 in pares_test:
    if w1 in w2i and w2 in w2i:
        s = sim_coseno_numpy(cooc_matrix[w2i[w1]], cooc_matrix[w2i[w2]])
        print(f"  {w1:12} <-> {w2:12} : {s:.4f}")
    else:
        print(f"  {w1:12} <-> {w2:12} : OOV")


=== Similitud Coseno con Vectores Crudos de Co-ocurrencia ===
  empresa      <-> mercado      : 0.5607
  gobierno     <-> país         : 0.8891
  banco        <-> financiero   : 0.8385
  empresa      <-> caballo      : OOV
  gobierno     <-> nube         : 0.3090


### Palabras con más co-ocurrencias directas
Se listan las diez palabras que más co-ocurren con empresa y con gobierno.

In [7]:
def top_cooc(palabra, topn=10):
    if palabra not in w2i:
        return []
    fila = cooc_matrix[w2i[palabra]].copy()
    fila[w2i[palabra]] = 0
    top_indices = np.argsort(fila)[::-1][:topn]
    return [(i2w[idx], int(fila[idx])) for idx in top_indices]

print("Top 10 co-ocurrencias directas con 'empresa':")
print(top_cooc("empresa"))
print("\nTop 10 co-ocurrencias directas con 'gobierno':")
print(top_cooc("gobierno"))


Top 10 co-ocurrencias directas con 'empresa':
[('la', 215), ('de', 168), ('una', 69), ('en', 61), ('que', 61), ('y', 50), ('a', 43), ('con', 35), ('el', 21), ('para', 20)]

Top 10 co-ocurrencias directas con 'gobierno':
[('el', 235), ('del', 114), ('de', 93), ('que', 75), ('y', 62), ('en', 42), ('la', 40), ('nacional', 36), ('por', 31), ('a', 30)]


¿Qué dominan los conteos crudos?
Las palabras funcionales, como la, de, el, del, en, que y y. Los conteos miden frecuencia y no asociación, así que cualquier palabra frecuente co-ocurre mucho con artículos y preposiciones por simple azar.

### 2.2 PPMI
PPMI compara la probabilidad conjunta de dos palabras con la que tendrían si fueran independientes, y deja en cero los valores negativos. Se implementa vectorizado con NumPy.

In [8]:
total_cooc = cooc_matrix.sum()
P_wc = cooc_matrix / total_cooc
p_w = cooc_matrix.sum(axis=1) / total_cooc
p_c = cooc_matrix.sum(axis=0) / total_cooc

denominador = np.outer(p_w, p_c)
mask = (P_wc > 0) & (denominador > 0)
ppmi_matrix = np.zeros_like(cooc_matrix)
ppmi_matrix[mask] = np.maximum(0, np.log2(P_wc[mask] / denominador[mask]))

def top_ppmi(palabra, topn=10):
    if palabra not in w2i:
        return []
    fila = ppmi_matrix[w2i[palabra]].copy()
    top_indices = np.argsort(fila)[::-1][:topn]
    return [(i2w[idx], float(fila[idx])) for idx in top_indices]

print("=== Top 10 Asociaciones con PPMI ===")
print("Top 10 PPMI con 'empresa':", [(w, round(v, 2)) for w, v in top_ppmi("empresa")])
print("\nTop 10 PPMI con 'gobierno':", [(w, round(v, 2)) for w, v in top_ppmi("gobierno")])


=== Top 10 Asociaciones con PPMI ===
Top 10 PPMI con 'empresa': [('ai', 5.31), ('telecomunicaciones', 5.28), ('líder', 5.16), ('llega', 4.91), ('colombiana', 4.76), ('especializada', 4.69), ('cualquier', 4.59), ('estadounidense', 4.4), ('hizo', 4.32), ('asegurar', 4.29)]

Top 10 PPMI con 'gobierno': [('corporativo', 7.07), ('federal', 5.71), ('petro', 5.17), ('nacional', 5.14), ('sánchez', 5.14), ('funcionarios', 5.01), ('iván', 4.86), ('estará', 4.54), ('mexicano', 4.46), ('llamado', 4.4)]


¿Qué cambió y por qué?
PPMI divide entre lo esperado por azar, así que las palabras muy frecuentes se castigan y las asociaciones específicas suben. Para empresa quedan telecomunicaciones, líder y colombiana; para gobierno, corporativo, federal, petro y funcionarios. Los conteos premian lo frecuente; PPMI premia lo específico.

### 2.3 TruncatedSVD a 100 dimensiones
Se reduce la matriz PPMI a 100 componentes y se guarda como KeyedVectors para usarla en la sección 5.

In [9]:
svd = TruncatedSVD(n_components=100, random_state=42)
embeddings_svd_matrix = svd.fit_transform(ppmi_matrix)

print(f"Forma de embeddings SVD: {embeddings_svd_matrix.shape}")
print(f"Varianza acumulada explicada (100 componentes): {svd.explained_variance_ratio_.sum() * 100:.2f}%")

kv_svd = KeyedVectors(vector_size=100)
kv_svd.add_vectors(top_2000_words, embeddings_svd_matrix)
ruta_svd = ruta_guardado("svd_ppmi_model.vec")
kv_svd.save_word2vec_format(ruta_svd)
print(f"Modelo SVD sobre PPMI guardado exitosamente en '{ruta_svd}'.")


Forma de embeddings SVD: (2000, 100)
Varianza acumulada explicada (100 componentes): 24.07%
Modelo SVD sobre PPMI guardado exitosamente en 'svd_ppmi_model.vec'.


### Por qué las filas ya son vectores, y qué resuelven PPMI y SVD
1. Cada fila describe los contextos de su palabra, y la hipótesis distribucional dice que eso es el significado. Por eso la fila ya es un vector, y la distancia entre filas mide afinidad.
2. PPMI quita el sesgo de frecuencia: las stopwords dejan de dominar y las colocaciones reales suben.
3. SVD comprime 2,000 columnas ralas en 100 dimensiones densas, y palabras que comparten contextos terminan cerca aunque nunca aparezcan juntas.

## 3. Investigación teórica

### 3.1 CBOW y skip-gram
CBOW predice la palabra central a partir del promedio de su contexto; skip-gram predice cada palabra del contexto a partir de la central. Skip-gram rinde más con palabras raras, porque cada aparición genera su propia actualización en lugar de diluirse en un promedio.

### 3.2 Negative sampling frente al softmax completo
Un softmax completo suma sobre todo el vocabulario en cada paso; negative sampling lo cambia por una clasificación binaria contra unas pocas palabras tomadas al azar. La probabilidad de que un par sea real es la sigmoide del producto punto entre el vector de la palabra central y el vector de la palabra de contexto. El costo depende de las negativas y no del vocabulario.

### 3.3 Las dos tablas de vectores
Word2Vec guarda una tabla para la palabra como centro y otra para la palabra como contexto, así ninguna palabra se predice a sí misma de forma trivial. Al final se conserva la tabla de palabras.

### 3.4 Word2Vec, GloVe y fastText
Word2Vec es predictivo y local: aprende ventana por ventana. GloVe ajusta conteos globales de co-ocurrencia. fastText suma vectores de n-gramas de caracteres. En español eso pesa porque un verbo tiene decenas de conjugaciones, y fastText reconoce la raíz en formas que nunca vio.

### 3.5 Parámetros de gensim.models.Word2Vec
- vector_size: dimensión de los vectores.
- window: alcance del contexto a cada lado.
- min_count: frecuencia mínima para entrar al vocabulario.
- sg: 1 para skip-gram y 0 para CBOW.
- negative: muestras negativas por par.
- sample: submuestreo de palabras muy frecuentes.
- epochs: pasadas sobre el corpus.

### 3.6 nn.Embedding
nn.Embedding es una tabla de filas aprendibles, con padding_idx para ignorar el relleno. from_pretrained carga vectores ya entrenados y con freeze los deja fijos. EmbeddingBag promedia o suma las filas de una secuencia en un solo paso. Multiplicar un vector one-hot por la matriz E devuelve la fila de su único uno, así que el producto y la búsqueda por índice dan lo mismo; la búsqueda solo es más barata.

## 4. Entrenar Word2Vec y FastText

### 4.1 Variantes de Word2Vec
Se entrena el modelo base y cinco variantes que cambian un solo parámetro: CBOW frente a skip-gram, ventana 2 frente a 10, y dimensión 50 frente a 300.

In [10]:
palabras_control = ["banco", "gobierno", "empresa", "mercado", "tecnología", "sostenibilidad"]

configuraciones = [
    ("Base (Skip-gram, w=5, d=100)", {"vector_size": 100, "window": 5, "min_count": 2, "sg": 1, "negative": 5, "epochs": 10}),
    ("Variante 1: CBOW (sg=0)", {"vector_size": 100, "window": 5, "min_count": 2, "sg": 0, "negative": 5, "epochs": 10}),
    ("Variante 2: Ventana corta (w=2)", {"vector_size": 100, "window": 2, "min_count": 2, "sg": 1, "negative": 5, "epochs": 10}),
    ("Variante 3: Ventana amplia (w=10)", {"vector_size": 100, "window": 10, "min_count": 2, "sg": 1, "negative": 5, "epochs": 10}),
    ("Variante 4: Dimensión baja (d=50)", {"vector_size": 50, "window": 5, "min_count": 2, "sg": 1, "negative": 5, "epochs": 10}),
    ("Variante 5: Dimensión alta (d=300)", {"vector_size": 300, "window": 5, "min_count": 2, "sg": 1, "negative": 5, "epochs": 10}),
]

modelos_w2v = {}
resultados_variantes = []

for nombre, params in configuraciones:
    t0 = time.time()
    m = Word2Vec(sentences=oraciones_noticias, seed=42, workers=1, **params)
    t_train = time.time() - t0
    modelos_w2v[nombre] = m
    
    vecinos_dict = {}
    for pc in palabras_control:
        if pc in m.wv:
            v_list = [w for w, s in m.wv.most_similar(pc, topn=5)]
            vecinos_dict[pc] = ", ".join(v_list)
        else:
            vecinos_dict[pc] = "N/A"
            
    resultados_variantes.append({
        "Configuración": nombre,
        "Tiempo (s)": round(t_train, 2),
        **vecinos_dict
    })

df_variantes = pd.DataFrame(resultados_variantes)
print("=== Comparativa de Variantes de Word2Vec en Noticias ===")
display(df_variantes[["Configuración", "Tiempo (s)", "banco", "gobierno", "sostenibilidad"]])


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

=== Comparativa de Variantes de Word2Vec en Noticias ===


,Configuración,Tiempo (s),banco,gobierno,sostenibilidad
0,"Base (Skip-gram, w=5, d=100)",11.73,"central, popular, agrario, gnb, finandina","congreso, consejo, intervenir, decreto, regulador","digitalización, estratégicas, palancas, actuac..."
1,Variante 1: CBOW (sg=0),3.75,"cordillera, checa, grupo, dominicana, entidad","congreso, estado, organismo, clima, paquete","innovación, digitalización, transformación, de..."
2,Variante 2: Ventana corta (w=2),6.80,"central, gnb, popular, sudameris, finandina","congreso, bce, federal, anuncio, intervenir","digitalización, emprendimiento, descarbonizaci..."
3,Variante 3: Ventana amplia (w=10),17.83,"central, agrario, república, finandina, popular","congreso, biden, intervenir, joe, voluntad","sociedad, prioridades, optimice, monográficos,..."
4,Variante 4: Dimensión baja (d=50),9.22,"central, agrario, popular, entidad, coopcentral","congreso, senado, decreto, consejo, regulador","digitalización, sociedad, creatividad, misión,..."
5,Variante 5: Dimensión alta (d=300),24.75,"central, popular, república, agrario, gnb","congreso, decreto, consejo, intervenir, plan","digitalización, estratégicas, diversidad, soci..."


¿Dan las ventanas pequeñas vecinos más sintácticos y las grandes más temáticos?
Sí. Con ventana 2, banco se rodea de gnb, sudameris y finandina, que ocupan su mismo lugar en la oración. Con ventana 10, gobierno se rodea de biden, joe y voluntad, que comparten su tema. Ventana corta, vecinos sintácticos; ventana amplia, vecinos temáticos.

### 4.2 Noticias frente al Quijote
Se entrena un modelo con la misma configuración base sobre el Quijote y se comparan cinco palabras.

In [11]:
w2v_quijote = Word2Vec(
    sentences=oraciones_quijote,
    vector_size=100, window=5, min_count=2,
    sg=1, negative=5, epochs=10, seed=42, workers=1
)

mejor_w2v = modelos_w2v["Base (Skip-gram, w=5, d=100)"]
ruta_w2v = ruta_guardado("mejor_modelo_word2vec.kv")
mejor_w2v.wv.save_word2vec_format(ruta_w2v)
print(f"Mejor modelo Word2Vec guardado en '{ruta_w2v}'.")

palabras_comp = ["señor", "dinero", "casa", "caballo", "palabra"]
filas_comp = []
for p in palabras_comp:
    vn = [w for w, s in mejor_w2v.wv.most_similar(p, topn=5)] if p in mejor_w2v.wv else []
    vq = [w for w, s in w2v_quijote.wv.most_similar(p, topn=5)] if p in w2v_quijote.wv else []
    filas_comp.append({
        "Palabra": p,
        "Vecinos en Noticias (Siglo XXI)": ", ".join(vn) if vn else "OOV (Frecuencia < 2)",
        "Vecinos en Don Quijote (Siglo XVII)": ", ".join(vq)
    })

df_comp = pd.DataFrame(filas_comp)
print("=== Comparación de Vecinos por Dominio y Época ===")
display(df_comp)


Mejor modelo Word2Vec guardado en 'mejor_modelo_word2vec.kv'.
=== Comparación de Vecinos por Dominio y Época ===


,Palabra,Vecinos en Noticias (Siglo XXI),Vecinos en Don Quijote (Siglo XVII)
0,señor,"sentí, fieles, coincidimos, conservadora, regu...","pedro, tarfe, perdóneme, álvaro, vuesa"
1,dinero,"estorbo, guardar, crediticio, trámite, permitía","dueño, pareciéndome, enviase, rogarle, suspensión"
2,casa,"pasiva, ininterrumpidas, terraza, compatibiliz...","aldea, mocedad, fiesta, compañía, hacienda"
3,caballo,OOV (Frecuencia < 2),"pie, freno, rodado, madera, albarda"
4,palabra,"inevitable, letras, tantos, agravado, reconfig...","hablase, respondelle, hablar, levantase, suspirar"


¿Qué dice esto sobre la frase el significado viene del corpus?
Que el vector no guarda una esencia de la palabra sino su uso en un corpus. Caballo se rodea de freno, rodado y albarda en el Quijote y ni aparece en las noticias. Empresa es negocio en unas y aventura en el otro.

### 4.3 FastText y palabras fuera del vocabulario
Se entrena FastText con la configuración base y se prueban tres palabras que Word2Vec no puede procesar.

In [12]:
ft_noticias = FastText(
    sentences=oraciones_noticias,
    vector_size=100, window=5, min_count=2,
    sg=1, negative=5, epochs=10, seed=42, workers=1
)

palabras_oov = [
    ("sostenibilidat", "Error de dedo (typo)"),
    ("desaceleráramos", "Conjugación verbal poco común"),
    ("criptoempresa", "Neologismo / inventada")
]

print("=== Evaluación de Palabras Fuera de Vocabulario (OOV) ===")
for w_oov, tipo in palabras_oov:
    print(f"\n-> Término: '{w_oov}' ({tipo})")
    # Prueba en Word2Vec
    if w_oov in mejor_w2v.wv:
        print("   Word2Vec: Encontrada.")
    else:
        print("   Word2Vec: KeyError — No puede generar vector ni vecinos.")
    # Prueba en FastText
    vec_ft = ft_noticias.wv[w_oov]
    sims = ft_noticias.wv.most_similar(w_oov, topn=4)
    print(f"   FastText: Vector inferido exitosamente ({vec_ft.shape}). Vecinos más cercanos:")
    for sim_w, sc in sims:
        print(f"     * {sim_w:18}: {sc:.4f}")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


=== Evaluación de Palabras Fuera de Vocabulario (OOV) ===

-> Término: 'sostenibilidat' (Error de dedo (typo))
   Word2Vec: KeyError — No puede generar vector ni vecinos.
   FastText: Vector inferido exitosamente ((100,)). Vecinos más cercanos:
     * sostenibilidad    : 0.9675
     * sostenida         : 0.8435
     * sostenible        : 0.7967
     * sostener          : 0.7762

-> Término: 'desaceleráramos' (Conjugación verbal poco común)
   Word2Vec: KeyError — No puede generar vector ni vecinos.
   FastText: Vector inferido exitosamente ((100,)). Vecinos más cercanos:
     * desaceleraría     : 0.8912
     * desacelere        : 0.8901
     * desacelerándose   : 0.8873
     * desaceleró        : 0.8801

-> Término: 'criptoempresa' (Neologismo / inventada)
   Word2Vec: KeyError — No puede generar vector ni vecinos.
   FastText: Vector inferido exitosamente ((100,)). Vecinos más cercanos:
     * empresa           : 0.8107
     * criptografía      : 0.7789
     * criptomoneda      : 0.7

### Mejor modelo propio
Se elige el modelo base, skip-gram con ventana 5 y 100 dimensiones. Sus vecinos de banco son los más limpios y equilibra mejor lo sintáctico y lo temático que CBOW o las dimensiones extremas. Queda guardado en Lab#9/mejor_modelo_word2vec.kv.

## 5. Similitud, vecinos y analogías

Se comparan cuatro modelos: SVD sobre PPMI, el mejor Word2Vec de noticias, el Word2Vec del Quijote y fastText SUC con 100 dimensiones y 100,000 vectores.

In [13]:
# Cargar fastText SUC pre-entrenado
ruta_suc = resolver_ruta("embeddings-m-model.vec", ["Lab#9/embeddings-m-model.vec", "../Lab#9/embeddings-m-model.vec"])
print(f"Cargando fastText SUC desde: {ruta_suc}...")
suc = KeyedVectors.load_word2vec_format(ruta_suc, limit=100_000)

# 5.1 Verificación de función propia de coseno
pares_verif = [("banco", "dinero"), ("gobierno", "país"), ("empresa", "mercado")]
print("=== Verificación Coseno Propio vs Gensim ===")
for p1, p2 in pares_verif:
    v_propio = sim_coseno_numpy(mejor_w2v.wv[p1], mejor_w2v.wv[p2])
    v_gensim = float(mejor_w2v.wv.similarity(p1, p2))
    print(f"  {p1:10} <-> {p2:10} | Propio: {v_propio:.6f} | Gensim: {v_gensim:.6f} | Error: {abs(v_propio - v_gensim):.2e}")


Cargando fastText SUC desde: embeddings-m-model.vec...
=== Verificación Coseno Propio vs Gensim ===
  banco      <-> dinero     | Propio: 0.217331 | Gensim: 0.217331 | Error: 4.47e-08
  gobierno   <-> país       | Propio: 0.322145 | Gensim: 0.322145 | Error: 0.00e+00
  empresa    <-> mercado    | Propio: 0.310992 | Gensim: 0.310992 | Error: 0.00e+00


In [14]:
palabras_eval_4m = ["banco", "gobierno", "empresa", "mercado", "trabajo"]
modelos_4 = [
    ("SVD sobre PPMI", kv_svd),
    ("Word2Vec Noticias", mejor_w2v.wv),
    ("Word2Vec Quijote", w2v_quijote.wv),
    ("fastText SUC", suc)
]

filas_vecinos = []
for p in palabras_eval_4m:
    fila = {"Palabra": p}
    for m_nombre, m_kv in modelos_4:
        if p in m_kv:
            vecs = [w for w, s in m_kv.most_similar(p, topn=5)]
            fila[m_nombre] = ", ".join(vecs)
        else:
            fila[m_nombre] = "OOV"
    filas_vecinos.append(fila)

df_vecinos_4m = pd.DataFrame(filas_vecinos)
print("=== Top 5 Vecinas en los Cuatro Modelos ===")
display(df_vecinos_4m)


=== Top 5 Vecinas en los Cuatro Modelos ===


,Palabra,SVD sobre PPMI,Word2Vec Noticias,Word2Vec Quijote,fastText SUC
0,banco,"davivienda, caja, bbva, bancolombia, colombia","central, popular, agrario, gnb, finandina","púsosela, acomodóse, desenvainó, corva, túnica","hipotecario, bancos, bank, citibank, monetario"
1,gobierno,"federal, central, congreso, mercado, regulador","congreso, consejo, intervenir, decreto, regulador","condado, consejo, pago, cobrar, regalo","ministro, gobiernos, mandatario, bierno, golpista"
2,empresa,"plataforma, compañía, tecnología, entidad, marca","startup, compañía, paletería, criptomoneda, paack","peligrosa, herida, visión, demanda, ganase","compañía, constructora, subsidiaria, asegurado..."
3,mercado,"sector, sistema, país, mundo, colombiano","sector, apetito, accionario, colombiano, solve...",OOV,"mer­cado, mercados, cado, minorista, produc­"
4,trabajo,"también, talento, profesional, investigación, ...","remoto, teletrabajo, colaborativo, riqueza, ar...","recogimiento, poblado, alivio, conocimiento, c...","remunerado, empleo, detrabajo, tra­bajo, trabajos"


### Antónimos
Se calcula la similitud en cuatro pares de antónimos.

In [15]:
pares_ant = [("bueno", "malo"), ("subir", "bajar"), ("ganar", "perder"), ("público", "privado")]
filas_ant = []
for a1, a2 in pares_ant:
    fila = {"Par Antónimo": f"{a1} / {a2}"}
    for m_nombre, m_kv in modelos_4:
        if a1 in m_kv and a2 in m_kv:
            fila[m_nombre] = round(float(m_kv.similarity(a1, a2)), 4)
        else:
            fila[m_nombre] = "OOV"
    filas_ant.append(fila)

df_ant = pd.DataFrame(filas_ant)
print("=== Similitud en Pares de Antónimos ===")
display(df_ant)


=== Similitud en Pares de Antónimos ===


,Par Antónimo,SVD sobre PPMI,Word2Vec Noticias,Word2Vec Quijote,fastText SUC
0,bueno / malo,OOV,0.7615,0.7995,0.7174
1,subir / bajar,OOV,0.8045,0.7876,0.9372
2,ganar / perder,OOV,0.6676,0.7495,0.6895
3,público / privado,0.5367,0.6761,OOV,0.6533


Las similitudes son altas, de 0.54 a 0.94. Bueno y malo aparecen en los mismos marcos, el precio fue bueno y el precio fue malo, y el modelo ve intercambiabilidad y no oposición. Similitud distribucional es compartir contexto; sinonimia es compartir significado.

### 5.2 Conjunto de similitud propio y correlación de Spearman
Se asignan puntajes de 0 a 10 a 20 pares, que incluyen pares muy parecidos, relacionados y sin relación, y se correlacionan con el coseno de cada modelo.

In [16]:
pares_oro = [
    ("empresa", "compañía", 9.5),
    ("banco", "entidad", 8.0),
    ("dinero", "capital", 8.5),
    ("gobierno", "estado", 8.0),
    ("presidente", "mandatario", 9.0),
    ("mercado", "comercio", 7.5),
    ("precio", "costo", 8.0),
    ("economía", "finanzas", 8.5),
    ("trabajo", "empleo", 9.0),
    ("país", "nación", 8.5),
    ("banco", "río", 2.0),
    ("mercado", "manzana", 3.0),
    ("empresa", "fútbol", 2.0),
    ("gobierno", "guitarra", 0.5),
    ("dinero", "felicidad", 4.0),
    ("casa", "edificio", 7.0),
    ("caballo", "animal", 8.0),
    ("computadora", "tecnología", 8.5),
    ("tiempo", "reloj", 7.5),
    ("nube", "lluvia", 7.0)
]

scores_humano = [p[2] for p in pares_oro]
resultados_spearman = []

for m_nombre, m_kv in modelos_4:
    valid_human = []
    valid_model = []
    oov_count = 0
    for w1, w2, score in pares_oro:
        if w1 in m_kv and w2 in m_kv:
            sim = float(m_kv.similarity(w1, w2))
            valid_human.append(score)
            valid_model.append(sim)
        else:
            oov_count += 1
            
    corr, p_val = spearmanr(valid_human, valid_model)
    resultados_spearman.append({
        "Modelo": m_nombre,
        "Pares Evaluados": len(valid_human),
        "Pares OOV": oov_count,
        "Correlación Spearman (ρ)": round(corr, 4),
        "p-valor": f"{p_val:.4e}"
    })

df_spearman = pd.DataFrame(resultados_spearman)
print("=== Correlación de Spearman con Juicio Humano ===")
display(df_spearman)


=== Correlación de Spearman con Juicio Humano ===


,Modelo,Pares Evaluados,Pares OOV,Correlación Spearman (ρ),p-valor
0,SVD sobre PPMI,9,11,-0.1725,6.5715e-01
1,Word2Vec Noticias,17,3,0.4474,7.1773e-02
2,Word2Vec Quijote,8,12,0.2410,5.6535e-01
3,fastText SUC,20,0,0.7421,1.7976e-04


fastText SUC se parece más a mi juicio, con rho de 0.74 y ningún par fuera del vocabulario. Word2Vec de noticias llega a 0.45 con tres pares fuera. SVD y el Quijote casi no se pueden evaluar, con 11 y 12 pares fuera, y su correlación no es significativa.

### 5.3 Analogías
La función analogia resuelve a es a b como c es a qué, sumando el vector de b y el de c y restando el de a, con vectores normalizados y sin contar a, b ni c como respuesta. Se evalúan 15 analogías de cinco tipos.

In [17]:
lista_analogias = [
    # Género
    ("rey", "reina", "hombre", "mujer", "Género"),
    ("actor", "actriz", "director", "directora", "Género"),
    ("héroe", "heroína", "rey", "reina", "Género"),
    # País - Capital
    ("francia", "parís", "españa", "madrid", "País–Capital"),
    ("colombia", "bogotá", "perú", "lima", "País–Capital"),
    ("argentina", "buenos", "chile", "santiago", "País–Capital"),
    # País - Gentilicio
    ("méxico", "mexicano", "chile", "chileno", "País–Gentilicio"),
    ("colombia", "colombiano", "perú", "peruano", "País–Gentilicio"),
    ("españa", "español", "francia", "francés", "País–Gentilicio"),
    # Singular - Plural
    ("empresa", "empresas", "mercado", "mercados", "Singular–Plural"),
    ("banco", "bancos", "precio", "precios", "Singular–Plural"),
    ("país", "países", "año", "años", "Singular–Plural"),
    # Tiempos Verbales
    ("subió", "subirá", "creció", "crecerá", "Tiempos Verbales"),
    ("compró", "comprará", "vendió", "venderá", "Tiempos Verbales"),
    ("ganó", "ganará", "perdió", "perderá", "Tiempos Verbales"),
]

def resolver_analogia(m_kv, a, b, c, k=5, excluir_consulta=True):
    if a not in m_kv or b not in m_kv or c not in m_kv:
        return None
    res = m_kv.most_similar(positive=[b, c], negative=[a], topn=k if excluir_consulta else k+3)
    if not excluir_consulta:
        # Vectores unitarios, igual que gensim: b - a + c
        vec_obj = m_kv.get_vector(b, norm=True) - m_kv.get_vector(a, norm=True) + m_kv.get_vector(c, norm=True)
        vec_obj = vec_obj / np.linalg.norm(vec_obj)
        sims = np.dot(m_kv.get_normed_vectors(), vec_obj)
        top_idx = np.argsort(sims)[::-1][:k]
        return [m_kv.index_to_key[i] for i in top_idx]
    return [w for w, s in res]

# Verificar coincidencia de función propia con gensim
verif_top5 = mejor_w2v.wv.most_similar(positive=["empresas", "mercado"], negative=["empresa"], topn=5)
print("Verificación de most_similar de gensim:")
print(verif_top5)


Verificación de most_similar de gensim:
[('firmas', 0.44435733556747437), ('compañías', 0.4412662088871002), ('apuestas', 0.4254561960697174), ('favorables', 0.41350042819976807), ('tecnológicas', 0.4120541214942932)]


In [18]:
res_analogias = []
t1_acc = {m[0]: 0 for m in modelos_4}
t5_acc = {m[0]: 0 for m in modelos_4}
totales = {m[0]: 0 for m in modelos_4}

for a, b, c, target, cat in lista_analogias:
    fila = {"Tipo": cat, "Analogía": f"{a}:{b}::{c}:?", "Esperado": target}
    for m_nombre, m_kv in modelos_4:
        top_k = resolver_analogia(m_kv, a, b, c, k=5, excluir_consulta=True)
        if top_k is None:
            fila[m_nombre] = "OOV"
        else:
            totales[m_nombre] += 1
            if target in top_k:
                rank = top_k.index(target) + 1
                fila[m_nombre] = f"Top {rank} ({top_k[0]})"
                t5_acc[m_nombre] += 1
                if rank == 1:
                    t1_acc[m_nombre] += 1
            else:
                fila[m_nombre] = f"Fallo ({top_k[0]})"
    res_analogias.append(fila)

df_ana = pd.DataFrame(res_analogias)
print("=== Tabla de Resultados de Analogías ===")
display(df_ana[["Tipo", "Analogía", "Esperado", "Word2Vec Noticias", "fastText SUC"]])

print("\n=== Porcentaje de Aciertos por Modelo ===")
for m_nombre in [m[0] for m in modelos_4]:
    tot = totales[m_nombre]
    p1 = (t1_acc[m_nombre] / tot * 100) if tot else 0
    p5 = (t5_acc[m_nombre] / tot * 100) if tot else 0
    print(f"  {m_nombre:20}: Top 1 = {t1_acc[m_nombre]}/{tot} ({p1:.1f}%) | Top 5 = {t5_acc[m_nombre]}/{tot} ({p5:.1f}%)")


=== Tabla de Resultados de Analogías ===


,Tipo,Analogía,Esperado,Word2Vec Noticias,fastText SUC
0,Género,rey:reina::hombre:?,mujer,Fallo (cuarentena),Top 4 (chica)
1,Género,actor:actriz::director:?,directora,Fallo (laura),Top 1 (directora)
2,Género,héroe:heroína::rey:?,reina,OOV,Fallo (cocaína)
3,País–Capital,francia:parís::españa:?,madrid,Fallo (fijado),Top 1 (madrid)
4,País–Capital,colombia:bogotá::perú:?,lima,Fallo (bancoomeva),Top 2 (arequipa)
5,País–Capital,argentina:buenos::chile:?,santiago,Fallo (completando),Fallo (aires)
6,País–Gentilicio,méxico:mexicano::chile:?,chileno,Top 4 (colombiano),Top 1 (chileno)
7,País–Gentilicio,colombia:colombiano::perú:?,peruano,Fallo (chileno),Top 1 (peruano)
8,País–Gentilicio,españa:español::francia:?,francés,Fallo (ibex),Top 1 (francés)
9,Singular–Plural,empresa:empresas::mercado:?,mercados,Fallo (firmas),Top 1 (mercados)



=== Porcentaje de Aciertos por Modelo ===
  SVD sobre PPMI      : Top 1 = 1/8 (12.5%) | Top 5 = 1/8 (12.5%)
  Word2Vec Noticias   : Top 1 = 0/12 (0.0%) | Top 5 = 1/12 (8.3%)
  Word2Vec Quijote    : Top 1 = 0/4 (0.0%) | Top 5 = 0/4 (0.0%)
  fastText SUC        : Top 1 = 10/15 (66.7%) | Top 5 = 12/15 (80.0%)


### Analogías sin excluir las palabras de la consulta
Se repiten cinco analogías dejando que a, b y c compitan en el ranking.

In [19]:
print("=== Analogías SIN Excluir Palabras de Consulta (fastText SUC) ===")
for a, b, c, target, cat in lista_analogias[:5]:
    top_no_excl = resolver_analogia(suc, a, b, c, k=5, excluir_consulta=False)
    print(f"  {a}:{b}::{c}:? -> Top 5 sin exclusión: {top_no_excl}")


=== Analogías SIN Excluir Palabras de Consulta (fastText SUC) ===
  rey:reina::hombre:? -> Top 5 sin exclusión: ['chica', 'pelirroja', 'hombre', 'mujer', 'muchacha']
  actor:actriz::director:? -> Top 5 sin exclusión: ['directora', 'subdirectora', 'coreógrafa', 'director', 'secretaria']
  héroe:heroína::rey:? -> Top 5 sin exclusión: ['heroína', 'cocaína', 'metanfetamina', 'anfetamina', 'hachís']
  francia:parís::españa:? -> Top 5 sin exclusión: ['madrid', 'sevilla', 'barcelona', 'parís', 'málaga']
  colombia:bogotá::perú:? -> Top 5 sin exclusión: ['arequipa', 'lima', 'huancayo', 'cuzco', 'cusco']


En héroe, heroína, rey, la palabra heroína queda en primer lugar. En las demás, hombre queda tercero, director cuarto y parís cuarto. La aritmética vectorial no es una igualdad; funciona porque se sacan del resultado las palabras de la pregunta.

## 6. Visualización del espacio

Se proyectan con PCA y con t-SNE palabras de seis grupos: países, meses, números, verbos de movimiento, términos económicos y profesiones. Solo entran las que existen en el mejor modelo propio y en fastText SUC.

In [20]:
grupos_palabras = {
    "Países": ["españa", "francia", "alemania", "italia", "méxico", "chile", "colombia", "argentina", "perú", "brasil", "china", "japón", "rusia", "portugal"],
    "Meses": ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"],
    "Números": ["uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez", "veinte", "cien", "mil", "millón"],
    "Verbos de movimiento": ["correr", "caminar", "viajar", "llegar", "salir", "entrar", "subir", "bajar", "avanzar", "partir", "regresar", "volver"],
    "Términos económicos": ["banco", "mercado", "empresa", "inflación", "inversión", "capital", "precio", "financiero", "crédito", "dinero", "tasa", "deuda", "bonos", "acciones"],
    "Profesiones": ["abogado", "médico", "ingeniero", "profesor", "director", "ministro", "presidente", "periodista", "juez", "economista", "científico", "escritor"]
}

palabras_visual = []
etiquetas_grupo = []
for grp, words in grupos_palabras.items():
    for w in words:
        if w in mejor_w2v.wv and w in suc:
            palabras_visual.append(w)
            etiquetas_grupo.append(grp)

X_w2v = np.array([mejor_w2v.wv[w] for w in palabras_visual])
X_suc = np.array([suc[w] for w in palabras_visual])

pca_w2v = PCA(n_components=2, random_state=42).fit_transform(X_w2v)
pca_suc = PCA(n_components=2, random_state=42).fit_transform(X_suc)

tsne_w2v = TSNE(n_components=2, random_state=42, perplexity=15, max_iter=1000).fit_transform(X_w2v)
tsne_suc = TSNE(n_components=2, random_state=42, perplexity=15, max_iter=1000).fit_transform(X_suc)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
coords_list = [
    (pca_w2v, "PCA 2D — Word2Vec Noticias (Propio)"),
    (pca_suc, "PCA 2D — fastText SUC (Preentrenado)"),
    (tsne_w2v, "t-SNE 2D — Word2Vec Noticias (Propio)"),
    (tsne_suc, "t-SNE 2D — fastText SUC (Preentrenado)")
]

palabras_destacadas = {"españa", "méxico", "china", "enero", "diciembre", "uno", "cien", "millón", "correr", "banco", "inflación", "médico", "director"}
palette = sns.color_palette("tab10", n_colors=len(grupos_palabras))

for ax, (coords, tit) in zip(axes.flatten(), coords_list):
    df_p = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "Grupo": etiquetas_grupo, "Palabra": palabras_visual})
    sns.scatterplot(data=df_p, x="x", y="y", hue="Grupo", palette=palette, s=70, alpha=0.85, ax=ax, edgecolor='k')
    for _, row in df_p.iterrows():
        if row["Palabra"] in palabras_destacadas:
            ax.annotate(row["Palabra"], (row["x"], row["y"]), xytext=(4, 4), textcoords='offset points', fontsize=8, fontweight='bold')
    ax.set_title(tit, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='best')

plt.tight_layout()
plt.show()


<Figure size 1600x1300 with 4 Axes>

### PCA y t-SNE
Los grupos se forman, pero no igual en los dos modelos. En fastText SUC con t-SNE los seis grupos quedan separados; con PCA se separan meses, números y verbos, y países, profesiones y términos económicos se tocan en el centro. En Word2Vec de noticias con t-SNE se agrupan países, meses, números y profesiones, mientras que términos económicos y verbos de movimiento se mezclan.

PCA conserva la varianza global; t-SNE conserva vecindarios. En t-SNE se puede confiar en los grupos pero no en la distancia entre ellos, porque el algoritmo estira y comprime el espacio para que los puntos no se apelmacen.

## 7. Análisis

### 7.1 SVD sobre PPMI frente a Word2Vec
Se parecen en parte: para banco ambos dan popular, finandina y gnb. SVD usa los conteos globales de forma exacta pero necesita una matriz de vocabulario por vocabulario; Word2Vec aprende ventana por ventana y escala a corpus enormes. GloVe junta ambos al ajustar vectores densos sobre los conteos globales.

### 7.2 Word2Vec frente a fastText SUC
La distancia es grande: rho de 0.45 contra 0.74, y en analogías 1 de 12 en top 5 contra 12 de 15. SUC aprendió de 2,600 millones de palabras y este modelo de 622,871 tokens. Con estos experimentos no se puede repartir la diferencia entre tamaño y método; el tamaño es la explicación más fuerte, pero queda sin demostrar.

### 7.3 Dónde funcionó la aritmética vectorial
Funcionó en gentilicios y en tiempos verbales, con 3 de 3 en SUC, y falló más en país y capital y en género, con 1 de 3 en top 1. El símbolo de aproximado no es una igualdad.

### 7.4 Transferencia entre dominios y épocas
El modelo hereda los significados de su corpus, y un modelo entrenado en un texto falla en otro, igual que el tagger del Laboratorio 6.

### 7.5 Embeddings frente a TF-IDF
Este notebook no incluye el experimento de clasificación, así que no hay cifras. Congelar la tabla conviene cuando los datos son pocos o los vectores vienen de un corpus grande; hacer fine-tuning conviene cuando hay muchos datos. Con el 10% de los datos, lo seguro es congelar.

### 7.6 La palabra banco
En los modelos banco solo trae vecinos financieros. Los otros sentidos, el asiento y el banco de peces, no aparecen porque el sentido más usado tapa a los demás dentro de un vector único. BERT resuelve esto con un vector distinto según la oración.

### 7.7 Auditoría antes de usarlos con personas
Primero los sesgos de género, comprobando si profesiones como ingeniero o director quedan más cerca de nombres masculinos, con pruebas como WEAT. Después los de nacionalidad y origen. Un sistema que decide sobre personas hereda los prejuicios del texto con que se entrenó.